# rounds

> The round protocol (speedup plan Part B): events starting in the same window are prepared on one snapshot, decided concurrently (the API calls) and applied in event order; with `d = 0` it is the event-driven loop exactly. Adapters give numpy members the three phases.

In [ ]:
#| default_exp rounds

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import heapq
from concurrent.futures import ThreadPoolExecutor

import numpy as np

from explRL.harness import MessageBus
from explRL.harness_agents import CommBandit
from explRL.multi import Outcome, Record, TeamHistory
from explRL.team_agents import PatchMember


def prepare(m, world, t):
    "Phase 1 (serial, agent order): read the world and channel at `t` and choose the input to the decision."
    if hasattr(m, 'prepare'):
        return m.prepare(world, t)
    if not isinstance(m, PatchMember):
        raise TypeError(f'{type(m).__name__} does not support rounds')
    m._world, m._t = world, t
    if isinstance(m, CommBandit) and isinstance(m.channel, MessageBus):
        for _, j, v, _ in m.channel.take(m.id):
            if m.w != 0:
                m._store()[j].append(v)
    obs = world.observe(m.id)
    return obs, m.act(obs)


def decide(m, prep):
    "Phase 2 (concurrent): the API call for LLM members; nothing for numpy members."
    return m.decide(prep) if hasattr(m, 'decide') else prep


def commit(m, world, t, dec):
    "Phase 3 (serial, event order): apply at the member's own start time, learn and post; -> Outcome."
    if hasattr(m, 'commit'):
        return m.commit(world, t, dec)
    obs, target = dec
    m._t = t
    reward, dt, loc, info = world.apply(m.id, target, t)
    m.learn(obs, target, reward, info)
    return Outcome(dt, info['kind'], loc, reward, (target, info['harvest'], info['nHere'], info['patch']))


def runRounds(members, world, d, turns=None, Ttot=None, t0=0.0, threads=8):
    """Rounds of window `d`. A round starts at the earliest pending start `s` and takes every member whose next event
    starts in [s, s + d) (with d = 0, only the earliest one, ties to the lower index: the event-driven loop). Their
    events are prepared one after another in agent order at `s`, decided concurrently, and applied in (start, index)
    order; an event a member starts later inside the same window is prepared and decided when its turn comes, in
    event order. Each member stops after `turns` events (int or list) or once its next start reaches t0 + Ttot."""
    n = len(members)
    budget = [int(turns)] * n if np.isscalar(turns) else (list(turns) if turns is not None else [None] * n)
    for i, m in enumerate(members):
        m.id = i
    hist = TeamHistory([m.extraNames for m in members])
    heap, done = [(t0, i) for i in range(n) if budget[i] is None or budget[i] > 0], [0] * n
    heapq.heapify(heap)

    def more(i, t):
        return (budget[i] is None or done[i] < budget[i]) and (Ttot is None or t < t0 + Ttot)

    with ThreadPoolExecutor(threads) as pool:
        while heap:
            s = heap[0][0]
            batch = [heapq.heappop(heap)] if d <= 0 else []
            while d > 0 and heap and heap[0][0] < s + d:
                batch.append(heapq.heappop(heap))
            batch.sort(key=lambda x: x[1])
            preps = [prepare(members[i], world, s) for _, i in batch]
            decs = list(pool.map(lambda a: decide(members[a[0][1]], a[1]), zip(batch, preps)))
            q = [(start, i, k) for k, (start, i) in enumerate(batch)]
            heapq.heapify(q)
            while q:
                start, i, k = heapq.heappop(q)
                m = members[i]
                dec = decs[k] if k >= 0 else decide(m, prepare(m, world, start))
                out = commit(m, world, start, dec)
                if not out.dt > 0:
                    raise ValueError(f'event duration must be > 0, got {out.dt}')
                hist.add(Record(i, start, out.dt, out.kind, out.location, out.reward, out.extras))
                done[i] += 1
                nxt = start + out.dt
                if not more(i, nxt):
                    continue
                if d > 0 and nxt < s + d:
                    heapq.heappush(q, (nxt, i, -1))          # later event in the same window: decided when it starts
                else:
                    heapq.heappush(heap, (nxt, i))
    return hist

## Tests

In [ ]:
import json, time, random
import numpy as np
from explRL.rounds import *
from explRL.harness import Blackboard, MessageBus
from explRL.harness_agents import CommBandit
from explRL.harness_runs import benchWorld, makeChannel, HarnessConfig
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_client import LLMClient
from explRL.llm_env import TextForagingEnv, locationNames
from explRL.llm_board import TextBoardEnv
from explRL.llm_fake import FakeClient, fakeReply
from explRL.multi import runTeam, spawnRngs
from explRL.taskboard import TaskBoardEnv
from explRL.board_agents import ClaimBoard

def same(a, b):
    return json.dumps(a, default=str) == json.dumps(b, default=str)

def numpyTeam(ch, N=6, seed=3):
    w = benchWorld(tStay=5.0); w.reset(seed, N)
    c = makeChannel(HarnessConfig(channel=ch), w.L, N, seed)
    return w, [CommBandit(c) for _ in range(N)]

# d = 0 is the event-driven loop exactly (numpy teams, by turns and by world time)
for ch in ('none', 'blackboard', 'messages'):
    w1, m1 = numpyTeam(ch); h1 = runTurns(m1, w1, 30)
    w2, m2 = numpyTeam(ch); h2 = runRounds(m2, w2, 0, turns=30)
    assert same(w1.ledger, w2.ledger) and same(h1.records, h2.records)
    w1, m1 = numpyTeam(ch); runTeam(m1, w1, 120.0)
    w2, m2 = numpyTeam(ch); runRounds(m2, w2, 0, Ttot=120.0, t0=0)      # runTeam starts at the integer 0
    assert same(w1.ledger, w2.ledger)

# ... and for LLM teams with the fake client (ledger and turn logs)
def llmTeam(task, N=4, seed=2, delay=0.0, jitter=False):
    def script(req):
        if delay or jitter:
            time.sleep(delay + (random.random() * 0.01 if jitter else 0))
        return fakeReply(req)
    if task == 'forage':
        w = benchWorld(tStay=5.0); w.reset(seed, N); env, ch = TextForagingEnv(w, locationNames(64, seed)), Blackboard(64, None, 0.01, N)
    else:
        w = TaskBoardEnv(seed, N, difficulty=1); env, ch = TextBoardEnv(w), ClaimBoard()
    fc = FakeClient(script)
    return w, [LLMMember(LLMClient('claude-haiku-5-5', client=fc), env, ch) for _ in range(N)]
for task in ('forage', 'board'):
    w1, m1 = llmTeam(task); runTurns(m1, w1, 6)
    w2, m2 = llmTeam(task); runRounds(m2, w2, 0, turns=6)
    assert same(w1.ledger, w2.ledger) and same([m.log for m in m1], [m.log for m in m2])

# rounds: every member makes exactly its turns; world time never goes backwards (foraging, mixed event lengths)
w, ms = llmTeam('forage', N=4)
h = runRounds(ms, w, 5.0, turns=8)
assert [len(h.forAgent(i)) for i in range(4)] == [8] * 4
assert all(a[1] <= b[1] for a, b in zip(w.ledger, w.ledger[1:]))
# in a round, members do not see same-round reports: the first decisions all see an empty board
assert all('Board' in m.log[0]['obs'] and 'empty' in m.log[0]['obs'] for m in ms)

# concurrency: a board round of 8 calls with a 0.1 s delay takes about one delay, not eight
w, ms = llmTeam('board', N=8, delay=0.1)
t0 = time.time(); runRounds(ms, w, 1.0, turns=2); wall = time.time() - t0
assert wall < 0.6, wall
w, ms = llmTeam('board', N=8, delay=0.1)
t0 = time.time(); runTurns(ms, w, 2); assert time.time() - t0 > 1.5

# results do not depend on thread timing
runs = []
for _ in range(3):
    w, ms = llmTeam('board', N=6, jitter=True)
    runRounds(ms, w, 1.0, turns=4)
    runs.append(json.dumps([w.ledger, [m.log for m in ms]], default=str))
assert len(set(runs)) == 1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()